In [1]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType 
from pyspark.sql import SparkSession
import logging

In [2]:
spark = ( 
 SparkSession
 .builder
 .master("local[*]")
 .appName('test')
 .getOrCreate()
)

26/04/17 16:20:20 WARN Utils: Your hostname, negan-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.41 instead (on interface enp42s0)
26/04/17 16:20:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 16:20:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 16:20:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
import json
from pyspark.sql.types import StructType, StructField, StringType

data = [
    (1, 5, 10, 15, 100),
    (2, 10, 15, 25, 200),
    (3, 12, 20, 35, 300),
    (4, 20, 25, 45, 400),
    (5, 20, 35, 45, 500)
]

colunas = ["NDOC", "INCOME_RELATED_AVERAGE", "INCOME_HOUSEHOLD_SUM", "HOUSEHOLD_QTY", "RELATED_QTY"]
df = spark.createDataFrame(data, colunas)

df.printSchema()
df.show(truncate=False)

root
 |-- NDOC: long (nullable = true)
 |-- INCOME_RELATED_AVERAGE: long (nullable = true)
 |-- INCOME_HOUSEHOLD_SUM: long (nullable = true)
 |-- HOUSEHOLD_QTY: long (nullable = true)
 |-- RELATED_QTY: long (nullable = true)



+----+----------------------+--------------------+-------------+-----------+
|NDOC|INCOME_RELATED_AVERAGE|INCOME_HOUSEHOLD_SUM|HOUSEHOLD_QTY|RELATED_QTY|
+----+----------------------+--------------------+-------------+-----------+
|1   |5                     |10                  |15           |100        |
|2   |10                    |15                  |25           |200        |
|3   |12                    |20                  |35           |300        |
|4   |20                    |25                  |45           |400        |
|5   |20                    |35                  |45           |500        |
+----+----------------------+--------------------+-------------+-----------+



"# BUILD CONDITION

In [6]:
from pyspark.sql import functions as spark_functions
from pyspark.sql.column import Column

class ConditionExpressionBuilder:
    def build_condition_expression(self, GV_features: dict) -> Column:
        """
        Constrói uma expressão Spark baseada na condição do JSON.
        Suporta AND, OR e operadores simples.
        """
        logging.warning(f"RUNNING BUILD CONDITION ====> {GV_features}")


        if GV_features["is-book"] == True:
            return self._run_book_rules()

        if GV_features["zera-score-se-tiver-filtro"] == True:
            return self._zerar_score()

        if GV_features["altera-retorno-se-tiver-filtro"] == True:
            return self._alterar_retornos()

        if GV_features["roda-flagmodels"] == True:
            return self._run_flagmodels()

        if GV_features["list-exception-bulkload"] == True:
            return self._run_filter_exception()

        if GV_features["BLAST_FULL"] == True:
            return self._run_BLAST_FULL_rules()

        return self._build_simple_condition(condition_definition)



    def _build_and_condition(self, list_of_conditions: list) -> Column:
        combined_condition = None

        for condition_item in list_of_conditions:
            current_condition = self.build_condition_expression(condition_item)

            if combined_condition is None:
                combined_condition = current_condition
            else:
                combined_condition = combined_condition & current_condition

        return combined_condition


    def _build_or_condition(self, list_of_conditions: list) -> Column:
        combined_condition = None

        for condition_item in list_of_conditions:
            current_condition = self.build_condition_expression(condition_item)
            if condition_item is None:
                combined_condition = current_condition

            else:
                combined_condition = combined_condition | current_condition

        return combined_condition


    def _build_simple_condition(self, condition_definition: dict) -> Column:
        column_name = condition_definition["column"]
        operator = condition_definition["op"]
        comparison_value = condition_definition.get("value")

        column_expression = spark_functions.col(column_name)

        if "cast" in condition_definition:
            column_expression =  column_expression.cast(condition_definition["cast"])

        if operator == "==":
            return column_expression == comparison_value

        if operator == "!=":
            return column_expression != comparison_value

        if operator == ">":
            return column_expression > comparison_value

        if operator == "<":
            return column_expression < comparison_value

        if operator == "isNull":
            return column_expression.isNull()

        if operator == "isNotNull":
            return column_expression.isNotNull()

        raise ValueError(f"Operador não suportado: {operator}")



# config loader

In [7]:
import json
from pathlib import Path
from urllib.parse import urlparse
import boto3

class ConfigurationLoader:
    def load_configuration_from_path(self, configuration_path: str) -> dict:
        if configuration_path.startswith("s3://"):
            return self._load_configuration_from_s3(configuration_path)

        return self._load_configuration_from_local_file(configuration_path)


    def _load_configuration_from_local_file(self, file_path: str) -> dict:
        with Path(file_path).open("r", encoding="utf-8") as file:
            return json.load(file)


    def _load_configuration_from_s3(self, s3_path: str) -> dict:
        parsed_path = urlparse(s3_path)
        bucket_name = parsed_path.netloc
        object_key = parsed_path.path.lstrip("/")

        s3_client = boto3.client("s3")
        response = s3_client.get_object(Bucket=bucket_name, Key=object_key)

        file_content = response["Body"].read().decode("utf-8")
        return json.loads(file_content)


# expression builder

In [8]:
from pyspark.sql import functions as spark_functions
from pyspark.sql.column import Column


class ColumnExpressionBuilder:
    def build_expression_from_rule_branch(self,rule_definition: dict, branch_definition: dict) -> Column:
        """ Constrói a expressão (then ou otherwise) """


        if "literal" in branch_definition:
            return spark_functions.lit(branch_definition["literal"])

        input_column_name = branch_definition.get(
            "input",
            rule_definition["input"]
        )

        current_expression = spark_functions.col(input_column_name)

        transformation_pipeline = branch_definition.get("pipeline", [])

        for transformation_step in transformation_pipeline:
            current_expression = self._apply_transformation_step(
                current_expression,
                transformation_step
            )

        return current_expression

    def _apply_transformation_step(
        self,
        expression: Column,
        transformation_step: dict
    ) -> Column:

        operation_type = transformation_step["op"]

        if operation_type == "cast":
            return expression.cast(transformation_step["type"])

        if operation_type == "lpad":
            return spark_functions.lpad(
                expression,
                transformation_step["len"],
                transformation_step["fill"]
            )

        if operation_type == "rpad":
            return spark_functions.rpad(
                expression,
                transformation_step["len"],
                transformation_step["fill"]
            )

        if operation_type == "substring":
            return spark_functions.substring(
                expression,
                transformation_step["start"],
                transformation_step["len"]
            )

        if operation_type == "trim":
            return spark_functions.trim(expression)

        if operation_type == "upper":
            return spark_functions.upper(expression)

        if operation_type == "lower":
            return spark_functions.lower(expression)

        raise ValueError(f"Transformação não suportada: {operation_type}")

# io manager

In [9]:
from pyspark.sql import SparkSession, DataFrame


class DataIOManager:
    def read_input_dataframe(self, spark_session: SparkSession, source_config: dict) -> DataFrame:
        data_format = source_config["format"]
        data_path = source_config["path"]

        dataframe_reader = spark_session.read.format(data_format)

        for option_key, option_value in source_config.get("read_options", {}).items():
            dataframe_reader = dataframe_reader.option(option_key, option_value)

        return dataframe_reader.load(data_path)

    def write_output_dataframe(self, dataframe: DataFrame, target_config: dict):
        data_format = target_config["format"]
        output_path = target_config["path"]
        write_mode = target_config.get("mode", "overwrite")

        dataframe_writer = dataframe.write.mode(write_mode).format(data_format)

        if "compression" in target_config:
            dataframe_writer = dataframe_writer.option(
                "compression",
                target_config["compression"]
            )

        dataframe_writer.save(output_path)

# rule engine


In [10]:
from pyspark.sql import DataFrame, functions as spark_functions


class DataTransformationEngine:
    def __init__(self, configuration: dict):
        self.configuration = configuration
        self.condition_builder = ConditionExpressionBuilder()
        self.expression_builder = ColumnExpressionBuilder()

    def apply_all_rules_to_dataframe(self, input_dataframe: DataFrame):
        """
        Aplica todas as regras do JSON no DataFrame
        """

        ordered_column_rules = self._get_column_rules_sorted_by_order()

        list_of_new_column_expressions = []

        for column_rule in ordered_column_rules:
            column_expression = self._build_column_expression(column_rule)

            aliased_expression = column_expression.alias(
                column_rule["output"]
            )

            list_of_new_column_expressions.append(aliased_expression)

        output_only = self.configuration.get("options", {}).get(
            "select_only_output_columns",
            False
        )

        if output_only:
            transformed_dataframe = input_dataframe.select(
                *list_of_new_column_expressions
            )
        else:
            transformed_dataframe = input_dataframe.select(
                "*",
                *list_of_new_column_expressions
            )

        ordered_output_columns = self._get_ordered_output_column_names()

        return transformed_dataframe, ordered_output_columns

    def _get_column_rules_sorted_by_order(self):
        return sorted(
            self.configuration["columns"],
            key=lambda rule: rule["order"]
        )

    def _get_ordered_output_column_names(self):
        return [
            rule["output"]
            for rule in self._get_column_rules_sorted_by_order()
        ]

    def _build_column_expression(self, column_rule: dict):
        """
        Monta a expressão final da coluna
        """

        then_expression = self.expression_builder.build_expression_from_rule_branch(
            column_rule,
            column_rule["then"]
        )

        if "condition" not in column_rule:
            return then_expression

        condition_expression = self.condition_builder.build_condition_expression(
            column_rule["condition"]
        )

        otherwise_definition = column_rule.get("otherwise")

        if otherwise_definition:
            otherwise_expression = self.expression_builder.build_expression_from_rule_branch(
                column_rule,
                otherwise_definition
            )
        else:
            otherwise_expression = spark_functions.col(column_rule["input"])

        return spark_functions.when(
            condition_expression,
            then_expression
        ).otherwise(otherwise_expression)

# job

In [11]:
from pyspark.sql import SparkSession


class DataTransformationJob:
    def __init__(self, spark_session: SparkSession, configuration: dict):
        self.spark_session = spark_session
        self.configuration = configuration
        self.engine = DataTransformationEngine(configuration)
        self.io_manager = DataIOManager()

    def execute(self):

        transformed_dataframe, ordered_columns = self.engine.apply_all_rules_to_dataframe(
            df
        )

        final_dataframe = transformed_dataframe.select(*ordered_columns)
        final_dataframe.show(truncate=False)
        self.io_manager.write_output_dataframe(
            final_dataframe,
            self.configuration["target"]
        )

# run

In [12]:
import argparse

from pyspark.sql import SparkSession

def create_spark_session(application_name: str) -> SparkSession:
    return (
        SparkSession.builder
        .appName(application_name)
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )


def main():

    loader = ConfigurationLoader()
    configuration = loader.load_configuration_from_path("/home/negan/projetos/data_bricks/data-project/hrhh.json")

    spark = create_spark_session(configuration["model"])

    job = DataTransformationJob(spark, configuration)
    job.execute()

    spark.stop()


if __name__ == "__main__":
    main()

26/04/16 23:20:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+---------+----------+-------+-------+-------+-------+--------+--------------+
|FIELD_1  |FIELD_2   |FIELD_3|FIELD_4|FIELD_5|FIELD_6|FIELD_7 |FIELD_8       |
+---------+----------+-------+-------+-------+-------+--------+--------------+
|100000000|0000005000|03     |29     |99999  |123    |12345-67|10000000000000|
|200000000|0000006000|04     |35     |99999  |456    |98765-43|20000000000000|
|300000000|6000000000|04     |35     |99999  |456    |98765-43|30000000000000|
|400000000|6000000000|04     |35     |99999  |456    |98765-43|40000000000000|
|500000000|6000000000|04     |35     |99999  |456    |98765-43|50000000000000|
|600000000|6000000000|04     |35     |99999  |456    |98765-43|60000000000000|
|700000000|6000000000|04     |35     |99999  |456    |98765-43|70000000000000|
|800000000|6000000000|04     |35     |99999  |456    |98765-43|17652208742   |
|900000000|6000000000|04     |35     |99999  |456    |98765-43|17652208742   |
|100000000|6000000000|04     |35     |99999  |456   